# PetGuard AI — Final interactive application

This notebook replaces the multi-cell zone editor workflow with **one Gradio app**.

**Workflow:** Upload fixed-camera MP4 → draw zone → Confirm → Process Video → review output → Edit/Redraw/Undo/Cancel → Confirm → Process again.

**No rerun of Colab cells is required** for zone edits or reprocessing. The app waits for an explicit click; there is no 90-second polling loop or background GPU processing.

**Important:** Commit the updated `src/app.py` to your GitHub repository before running this notebook. Your original midterm notebook and existing modules remain unchanged.


## 1. Clone latest GitHub source and install dependencies

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO = Path("/content/PetGuard-AI")
URL = "https://github.com/iamhuongthn/PetGuard-AI.git"
if REPO.exists() and (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "master"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "master"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/master"], check=True)
else:
    if REPO.exists():
        raise RuntimeError(f"{REPO} exists but is not a Git repository.")
    subprocess.run(["git", "clone", "--branch", "master", URL, str(REPO)], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO/"requirements.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gradio>=4.44,<6", "pillow"], check=True)
subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], check=True)
assert (REPO / "src/app.py").is_file(), "Push the ZIP's src/app.py to GitHub before running."
print("Ready: application loaded from GitHub.")


## 2. Launch PetGuard user interface (run once)

In [ ]:
import sys
sys.path.insert(0, str(REPO))
from src.app import build_app

app = build_app()
app.queue(default_concurrency_limit=1).launch(share=True, show_error=True, prevent_thread_lock=True)


## 3. Work entirely inside the app

1. Upload a **stationary-camera** MP4 (60 seconds / 250 MB maximum).
2. Select **Floor or room area** or **Sofa or furniture**.
3. Click at least 3 corners around the restricted region.
4. Click **Confirm zone**. You can Undo, Redraw, Edit, or Cancel without rerunning notebook cells.
5. Click **Process Video**. Wait for annotated MP4 and downloadable CSVs.
6. To change the zone: **Edit zone** or **Redraw** → **Confirm zone** → **Process Video** again.

Each run writes a separate folder under `results/runs/<run_id>/`, preventing accidental overwrites. Outputs in Colab are temporary; download or copy them to Drive before ending the session.

**Evaluation:** The app generates predictions, not ground truth. Use the original midterm data plan (Oxford-IIIT, COCO, PetGuard-specific labeled samples) to calculate recall, zone accuracy, and false violation rate. Human-as-dog false positives remain a known limitation.


## 4. Optional: Evaluate against manually verified ground truth

This preserves the evaluation checkpoint from your previous notebook. **Do not enter model confidence as accuracy.** Use manually labeled held-out samples, then calculate TP, TN, FP and FN for the zone-decision task. Your original midterm target was ≥90% zone accuracy and ≤10% false violation rate.


In [ ]:
# Replace these zeros with verified counts from your labeled evaluation set.
TP = TN = FP = FN = 0
n = TP + TN + FP + FN
if n == 0:
    print("No ground-truth evaluation entered yet.")
else:
    accuracy = (TP + TN) / n
    false_violation_rate = FP / (FP + TN) if (FP + TN) else float("nan")
    recall = TP / (TP + FN) if (TP + FN) else float("nan")
    print(f"Zone accuracy: {accuracy:.2%}")
    print(f"False violation rate: {false_violation_rate:.2%}")
    print(f"Violation recall: {recall:.2%}")
    print("Pet detection recall must be measured separately using annotated pet boxes.")
